In [5]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report
from datasets import DatasetDict, Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

In [19]:
#Setting the Datasets Files
train_path = "emotion_intensity_data/train.csv"  # Path to your training data
dev_path = "emotion_intensity_data/dev.csv"   # Path to development/validation data
test_path = "emotion_intensity_data/test.csv"    # Path to test data

def load_dataset(file_path):
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded {file_path} with {len(df)} samples")
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None

# Load all datasets
train_df = load_dataset(train_path)
dev_df = load_dataset(dev_path)
test_df = load_dataset(test_path)

Loaded emotion_intensity_data/train.csv with 2768 samples
Loaded emotion_intensity_data/dev.csv with 116 samples
Loaded emotion_intensity_data/test.csv with 2767 samples


In [ ]:
def prepare_labels(train_df, dev_df, test_df):
    """Create combined labels and mappings"""
    # Combine emotion and intensity (e.g., "anger_3")
    for df in [train_df, dev_df, test_df]:
        df['label'] = df['emotion'] + '_' + df['intensity'].astype(str)
    
    # Get all unique labels
    all_labels = set(train_df['label']).union(set(dev_df['label'])).union(set(test_df['label']))
    all_labels = sorted(list(all_labels))
    
    # Create mappings between labels and IDs
    label2id = {label: i for i, label in enumerate(all_labels)}
    id2label = {i: label for label, i in label2id.items()}
    
    print(f"Found {len(all_labels)} unique labels")
    return label2id, id2label

label2id, id2label = prepare_labels(train_df, dev_df, test_df)

Dataset Structure Verification:
Training data columns: ['id', 'text', 'anger', 'fear', 'joy', 'sadness', 'surprise']


KeyError: 'label'